## 06 — Latency and Aeron
Query recorded metric samples and Prometheus; validate units and injected delays.

In [ ]:
import clickhouse_connect, os
client = clickhouse_connect.get_client(
    host=os.environ.get('CLICKHOUSE_HOST', 'localhost'),
    port=int(os.environ.get('CLICKHOUSE_PORT', '8123')),
    username=os.environ.get('CLICKHOUSE_USER', 'default'),
    password=os.environ.get('CLICKHOUSE_PASSWORD', ''),
    database=os.environ.get('CLICKHOUSE_DATABASE', 'market'),
)
print('clickhouse', client.server_version)


In [ ]:
import os

# Scope to the run under test. `aeron_metrics` is a shared table: the
# in-cluster ingester keeps writing samples into it under its own run id while
# a local run is being verified, and samples from earlier runs persist. The
# cell near the end of this notebook asserts `seen == {run_id}`, which only
# holds if the query is scoped — so it always meant to filter and did not.
# Unscoped, it read other runs' samples and reported *their* inconsistencies.
run_id = os.environ.get('ERGO_RUN_ID', '')
scope = f"WHERE run_id = {int(run_id)}" if run_id.isdigit() else ""
samples = client.query(
    "SELECT captured_at_ns, run_id, recording_id, recording_start_position, "
    "recording_stop_position, replay_position, rows_ingested "
    f"FROM aeron_metrics {scope} ORDER BY captured_at_ns"
).result_rows
print('samples:', len(samples), samples[:3])
assert samples, 'no aeron metric samples recorded'

# Sampling must make progress: a frozen sampler is a dead sampler.
#
# The cursor check is scoped *per recording*, not applied to the table. This
# table accumulates every run, and neither column is monotone across it:
# `replay_position` restarts at 0 for each new recording (the recorders publish
# a fresh one every time they restart, and the long-lived ingester picks each
# up), and `rows_ingested` is a per-process counter that restarts with the
# ingester. Within a single recording the cursor must never move backwards.
by_recording = {}
for row in samples:
    by_recording.setdefault(row[2], []).append(row[5])
for rec, positions in sorted(by_recording.items()):
    assert positions == sorted(positions), (
        f'replay position went backwards within recording {rec}'
    )

positions = [row[5] for row in samples]
rows = [row[6] for row in samples]
assert max(positions) > min(positions), 'replay position never advanced'
assert max(rows) > 0, 'no rows ingested during the sampled run'
assert samples[0][3] < samples[0][4], 'recording stop position is not after its start'

run_id = os.environ.get('ERGO_RUN_ID', '')
if run_id.isdigit():
    seen = {row[1] for row in samples}
    assert seen == {int(run_id)}, f'samples from another run: {sorted(seen)}'
print('aeron metric samples verified for run', run_id or '(all runs)')

In [ ]:
import json, urllib.request

# The exporter is the archive-agent's own /metrics surface. Prometheus is not
# in the local fixture stack, so scrape the exporter directly rather than
# fabricating a Prometheus query result.
url = os.environ.get('ERGO_METRICS_URL', '')
assert url, 'ERGO_METRICS_URL not set; the exporter scrape is an outstanding criterion, not a pass'
with urllib.request.urlopen(url, timeout=10) as resp:
    body = resp.read().decode()
series = {
    line.split(' ', 1)[0]: float(line.split(' ', 1)[1])
    for line in body.splitlines()
    if line and not line.startswith('#')
}
print('exporter series:', series)
for name in (
    'ergo_agent_registrations_total',
    'ergo_agent_registrations_ok_total',
    'ergo_agent_registrations_failed_total',
    'ergo_agent_up',
):
    assert name in series, f'{name} missing from {url}'
assert series['ergo_agent_registrations_total'] == (
    series['ergo_agent_registrations_ok_total'] + series['ergo_agent_registrations_failed_total']
), 'registration counters do not reconcile'
assert series['ergo_agent_up'] == 1, 'agent reports not up'
print('exporter scrape verified:', url)